
<div style="background-color:#1F3864; padding:18px; border-radius:6px;">
<h1 style="color:#FFFFFF; font-family:Arial; margin:0;">Day 198 — ResumeGapAI: Core Engine Build</h1>
<p style="color:#D9E2F3; font-family:Arial; margin:6px 0 0 0;">Month 12 · Second Starred/Deployed GitHub Project (Days 197–200), Day 2 of 4 · Suggested repo: ResumeGapAI</p>
</div>



### Where this fits
Day 197 locked the idea: **ResumeGapAI** — TF-IDF similarity between a resume and a job description,
with a ranked gap-word report, scored 26.75 on the star-worthiness formula (tied with SchemaGuard,
won on personal expertise fit + real pain evidence).

**Today's job:** build and prove the core engine — the part with no UI, no deployment, just correct logic.
Day 199 wraps this in a polished Streamlit app + README + deploys it. Day 200 launches it.

**Scope lock for today (do not exceed):**
1. Text cleaning function
2. TF-IDF cosine similarity scorer — naive version, then a real fix you'll discover by testing it
3. Gap-word extractor (JD-important, resume-absent terms)
4. One wrapper function tying it together: `analyze_resume_gap()`
5. A Streamlit UI skeleton that calls that wrapper

No embeddings, no PDF parsing, no LLM calls today — those are optional Day 199 polish, not Day 198 core.
Keeping scope tight is itself the lesson: this is exactly the kind of scope creep that kills 3-day builds.



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 1 — Raw Data (LOCKED — do not modify)</h2>
</div>

Three resume/job-description pairs across three real-world roles you'd actually see on Freelancer.com
or LinkedIn. These are realistic synthetic text, not randomly generated — for text-similarity testing,
the actual wording matters more than a random seed would. Do not edit this cell; all cleaning happens
in later cells, on copies.


In [1]:
# RAW DATA — locked resume/JD pairs for Day 198. Do not modify this cell.

pairs = {
    "R1_DataAnalyst": {
        "resume": """
        Data Analyst with 2 years experience in Excel, SQL, and Power BI. Built dashboards for
        sales tracking and monthly reporting. Skilled in pivot tables, VLOOKUP, and basic Python
        for data cleaning using pandas. Experience with A/B testing analysis and customer segmentation
        using Excel. Familiar with Google Sheets and basic statistics.
        """,
        "jd": """
        We are looking for a Data Analyst with strong SQL and Python skills. Must have experience
        building interactive dashboards in Power BI or Tableau. Experience with pandas, statistical
        analysis, A/B testing, and machine learning is a plus. Knowledge of cloud platforms like AWS
        or Azure and experience with ETL pipelines is highly desirable. Strong communication skills
        for presenting insights to stakeholders required.
        """
    },
    "R2_DataScientist": {
        "resume": """
        Data Scientist skilled in Python, scikit-learn, and machine learning including classification,
        regression, and clustering. Experience with pandas, numpy, and matplotlib for exploratory
        data analysis. Built predictive models for churn prediction and customer segmentation.
        Familiar with Jupyter notebooks and Git version control. Basic exposure to SQL.
        """,
        "jd": """
        Seeking a Data Scientist with strong machine learning and deep learning skills. Must know
        TensorFlow or PyTorch, NLP techniques, and have experience deploying models to production
        using Docker and cloud platforms like AWS. Experience with MLOps, model monitoring, and
        A/B testing for model evaluation required. Strong Python and SQL skills essential.
        """
    },
    "R3_MLEngineer": {
        "resume": """
        Machine Learning Engineer with experience in Python, TensorFlow, and deploying models using
        Docker and Flask APIs. Built end-to-end ML pipelines including feature engineering, model
        training, and monitoring. Experience with AWS, CI/CD, and MLOps practices. Strong background
        in scikit-learn and PyTorch for production-grade model development.
        """,
        "jd": """
        Looking for a Machine Learning Engineer to build and deploy production ML systems. Requires
        strong Python, TensorFlow or PyTorch, Docker, Kubernetes, and cloud experience (AWS/GCP).
        Experience with MLOps, model monitoring, CI/CD pipelines, and A/B testing frameworks required.
        Knowledge of distributed training and model optimization is a plus.
        """
    }
}

for k, v in pairs.items():
    print(k, "-> resume chars:", len(v["resume"]), "| jd chars:", len(v["jd"]))

R1_DataAnalyst -> resume chars: 375 | jd chars: 466
R2_DataScientist -> resume chars: 393 | jd chars: 392
R3_MLEngineer -> resume chars: 384 | jd chars: 386



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 2 — Concept Notes</h2>
</div>

**TF-IDF cosine similarity, quick refresher (Month 6-7 + Month 9):** each document becomes a vector where
each dimension is a word, weighted by term-frequency × inverse-document-frequency. Cosine similarity
measures the angle between two such vectors — 100 means identical direction (same weighted vocabulary),
0 means no shared weighted terms.

**The gotcha this project will force you to confront:** `TfidfVectorizer().fit_transform([resume, jd])`
fits IDF on **only these two documents**. With a 2-document corpus, IDF collapses to near-meaningless
values (a word in both docs gets `idf = ln(2/2) + 1 = 1`; a word in only one gets `idf = ln(2/1) + 1 ≈ 1.69`).
That's barely any spread — so the "TF-IDF" score ends up behaving almost like plain term-overlap, and
real similarity gets **systematically underestimated**. This is the same discipline as the Day 189
carrier-transit-times lesson: don't trust what a method is *supposed* to do — verify what it *actually*
does on your data. You will prove this yourself in Task 2 vs Task 3 below.

**The fix used in production gap-analysis tools:** fit the vectorizer on a **larger background corpus**
of representative text (generic resume/JD language) so IDF reflects true rarity across a realistic
vocabulary, then transform just the resume/JD pair for scoring. This is a real architectural decision,
not a toy exercise — it's exactly the kind of judgment call that separates a tutorial clone from a
genuinely useful tool, which is what the Day 197 star-worthiness formula rewarded ResumeGapAI for.

**Gap words:** words with high TF-IDF weight in the JD but zero weight in the resume. These are the
terms a candidate should consider adding (if truthfully applicable) or preparing to discuss in an
interview. This is the single most useful output of the whole tool — the similarity score is a vanity
number, the gap list is the actionable one.

**Why no embeddings today:** sentence-transformers models (~80MB+ download, plus PyTorch as a dependency)
add real weight for an MX130/8GB machine and for Streamlit Cloud's free tier. TF-IDF is free, deploys
instantly, and — as you'll see in the answer key — after the background-corpus fix, gives a directionally
correct, explainable score. Embedding similarity is a legitimate Day 199 "if time permits" bonus, not
today's core.



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Practice Guide — Tasks (25 pts each unless noted)</h2>
</div>

Build these as functions in this notebook first (to test/prove them), then copy the final versions into
`resumegap_engine.py` (skeleton provided separately) — that file is what Day 199's Streamlit app will import.



**Task 1 — `clean_text(text)` (15 pts)**
Lowercase, strip everything except letters/digits/spaces, collapse whitespace. Test it on `pairs["R1_DataAnalyst"]["resume"]` and print the result.


In [2]:
# TASK 1: clean_text
# Goal: Lowercase, keep only letters/digits/spaces, collapse whitespace.
# Method: Use re.sub with pattern [^a-z0-9\s] to replace with space, then collapse \s+.

import re

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # single backslash for whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaned R1 resume:\n", clean_text(pairs["R1_DataAnalyst"]["resume"]))

Cleaned R1 resume:
 data analyst with 2 years experience in excel sql and power bi built dashboards for sales tracking and monthly reporting skilled in pivot tables vlookup and basic python for data cleaning using pandas experience with a b testing analysis and customer segmentation using excel familiar with google sheets and basic statistics



**Task 2 — `tfidf_similarity_naive(resume_text, jd_text)` (20 pts)**
Fit `TfidfVectorizer` jointly on `[clean_text(resume), clean_text(jd)]` only (2-document corpus).
Return the cosine similarity as a 0-100 score, rounded to 2 decimals. Run it on all 3 pairs and print
the scores. **Do not look at Section 4 yet** — you need to see these naive numbers with your own eyes
before Task 3 makes sense.


In [3]:
# TASK 2: tfidf_similarity_naive
# Goal: Fit TfidfVectorizer on only the resume and JD (2 docs), return similarity as 0-100.
# Method: Use TfidfVectorizer with default token pattern and stop_words='english'.

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_similarity_naive(resume_text: str, jd_text: str):
    docs = [clean_text(resume_text), clean_text(jd_text)]
    vectorizer = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    tfidf_matrix = vectorizer.fit_transform(docs)
    sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(sim * 100, 2)

print("\n--- Naive TF-IDF Scores ---")
for key, pair in pairs.items():
    print(key, tfidf_similarity_naive(pair["resume"], pair["jd"]))


--- Naive TF-IDF Scores ---
R1_DataAnalyst 22.02
R2_DataScientist 15.65
R3_MLEngineer 42.77



**Task 3 — `tfidf_similarity_fixed(resume_text, jd_text, background_docs)` (25 pts)**
This is the real engineering task. Re-read the Concept Notes gotcha above. Build a version that fits
the vectorizer on `background_docs + [resume, jd]`, but computes cosine similarity only between the
resume and JD rows (the last two rows of the fitted matrix). Use the 10-sentence `background_corpus`
provided below (locked — represents generic resume/JD vocabulary, not this project's actual test data).

In a markdown cell below your code, answer in your own words: **why does adding unrelated background
text change the JD-resume similarity score, when neither the resume nor JD text itself changed?**
This interpretation is scored separately (Task 3b, 10 pts) — it must explain the actual mechanism
(what IDF does with a bigger corpus), not just restate that "the fix worked."


In [4]:
# RAW DATA — locked background corpus (generic resume/JD language, not project test data). Do not modify.
background_corpus = [
    "experienced professional seeking growth opportunities in a fast paced environment",
    "responsible for managing projects and delivering results on time and within budget",
    "strong communication and collaboration skills working with cross functional teams",
    "proficient in Microsoft Office including Excel Word and PowerPoint for reporting",
    "bachelor degree in relevant field with several years of professional experience",
    "seeking a motivated candidate with strong analytical and problem solving skills",
    "must have excellent written and verbal communication skills and attention to detail",
    "experience working in an agile environment with sprint planning and stand ups",
    "candidate should have strong leadership skills and experience managing a team",
    "looking for someone who can work independently and also collaborate with stakeholders",
]

# TASK 3: tfidf_similarity_fixed
# Goal: Fit on background + resume + jd, but score only resume vs jd.
# Method: Build docs = [clean(bg) for bg in background_docs] + [clean(resume), clean(jd)],
#         fit vectorizer, take rows -2 and -1 for similarity.

def tfidf_similarity_fixed(resume_text: str, jd_text: str, background_docs: list):
    cleaned_bg = [clean_text(d) for d in background_docs]
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    all_docs = cleaned_bg + [cleaned_resume, cleaned_jd]
    vectorizer = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    tfidf_matrix = vectorizer.fit_transform(all_docs)
    sim = cosine_similarity(tfidf_matrix[-2:-1], tfidf_matrix[-1:])[0][0]
    return round(sim * 100, 2)

print("\n--- Fixed TF-IDF Scores ---")
for key, pair in pairs.items():
    naive = tfidf_similarity_naive(pair["resume"], pair["jd"])
    fixed = tfidf_similarity_fixed(pair["resume"], pair["jd"], background_corpus)
    print(key, "naive:", naive, "fixed:", fixed)


--- Fixed TF-IDF Scores ---
R1_DataAnalyst naive: 22.02 fixed: 27.83
R2_DataScientist naive: 15.65 fixed: 21.06
R3_MLEngineer naive: 42.77 fixed: 49.97


### Task 3b — Interpretation (with evidence from my own run)

Adding unrelated background text changes the JD–resume similarity score because **TF‑IDF weights are computed relative to the entire corpus** used to fit the vectorizer, not just the two documents being compared.

In the naive version (only the resume and JD), the IDF component is almost constant:
- A word in both docs gets `idf = ln(2/2) + 1 = 1`.
- A word in only one gets `idf = ln(2/1) + 1 ≈ 1.69`.

This narrow range means the score is essentially term overlap – common and rare terms get nearly the same weight. With the **background corpus**, IDF spreads: common words get lower weight, rare terms get higher weight.

**Concrete evidence from my own run (after fixing the code):**  
- R1_DataAnalyst: naive = 22.02 → fixed = 27.83 (delta = +5.81)  
- R2_DataScientist: naive = 15.65 → fixed = 21.06 (delta = +5.41)  
- R3_MLEngineer: naive = 42.77 → fixed = 49.97 (delta = +7.20)  

The fixed scores are higher because rare technical terms shared between the resume and JD (e.g., "TensorFlow", "Docker") now carry more weight. In R3, the score crosses the Weak→Moderate boundary (49.97 > 45), proving the fix changes the match classification.

In short: **the reference vocabulary changed** – and that changes the importance of each word, which shifts the cosine similarity.


**Task 4 — `gap_words(resume_text, jd_text, background_docs, top_n=8)` (15 pts)**
Using the fixed (background-corpus) TF-IDF fit, find words with non-zero JD weight and zero resume
weight, sort descending by JD weight, and return the top `top_n` as a list of `(word, weight)` tuples.
Run on all 3 pairs.


In [5]:
# TASK 4: gap_words
# Goal: Identify words important in JD but absent in resume.
# Method: Use same fit as fixed similarity; get feature names, rows for resume (index -2) and jd (-1);
#         keep features where jd_weight>0 and resume_weight==0, sort descending by jd_weight.

def gap_words(resume_text: str, jd_text: str, background_docs: list, top_n: int = 8):
    cleaned_bg = [clean_text(d) for d in background_docs]
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    all_docs = cleaned_bg + [cleaned_resume, cleaned_jd]
    vectorizer = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
    tfidf_matrix = vectorizer.fit_transform(all_docs)
    feature_names = vectorizer.get_feature_names_out()
    resume_row = tfidf_matrix[-2].toarray().flatten()
    jd_row = tfidf_matrix[-1].toarray().flatten()
    gaps = [(w, round(jd_row[i], 4)) for i, w in enumerate(feature_names) if jd_row[i] > 0 and resume_row[i] == 0]
    gaps.sort(key=lambda x: x[1], reverse=True)
    return gaps[:top_n]

print("\n--- Gap Words (top 8) ---")
for key, pair in pairs.items():
    print(key, gap_words(pair["resume"], pair["jd"], background_corpus))


--- Gap Words (top 8) ---
R1_DataAnalyst [('strong', np.float64(0.2262)), ('skills', np.float64(0.2051)), ('aws', np.float64(0.1661)), ('azure', np.float64(0.1661)), ('building', np.float64(0.1661)), ('cloud', np.float64(0.1661)), ('desirable', np.float64(0.1661)), ('etl', np.float64(0.1661))]
R2_DataScientist [('model', np.float64(0.3294)), ('strong', np.float64(0.2243)), ('skills', np.float64(0.2034)), ('aws', np.float64(0.1647)), ('cloud', np.float64(0.1647)), ('deep', np.float64(0.1647)), ('deploying', np.float64(0.1647)), ('docker', np.float64(0.1647))]
R3_MLEngineer [('build', np.float64(0.1785)), ('cloud', np.float64(0.1785)), ('deploy', np.float64(0.1785)), ('distributed', np.float64(0.1785)), ('frameworks', np.float64(0.1785)), ('gcp', np.float64(0.1785)), ('knowledge', np.float64(0.1785)), ('kubernetes', np.float64(0.1785))]



**Task 5 — `analyze_resume_gap(resume_text, jd_text, background_docs, top_n=8)` (15 pts)**
Wrapper that ties it all together. Returns a dict:
```python
{
    "similarity_score": float,      # from tfidf_similarity_fixed
    "match_band": str,               # "Strong Match" >=70, "Moderate Match" >=45, else "Weak Match"
    "gap_words": list,                # from gap_words()
}
```
This is the exact function `app.py` will import and call — its signature must not change without
also updating the Streamlit skeleton.


In [6]:
# TASK 5: analyze_resume_gap
# Goal: Wrapper to produce similarity, band, and gap words in one dict.
# Method: Use fixed similarity, match_band helper, and gap_words; return dict.

def match_band(score: float) -> str:
    if score >= 70:
        return "Strong Match"
    elif score >= 45:
        return "Moderate Match"
    else:
        return "Weak Match"

def analyze_resume_gap(resume_text: str, jd_text: str, background_docs: list, top_n: int = 8) -> dict:
    score = tfidf_similarity_fixed(resume_text, jd_text, background_docs)
    band = match_band(score)
    gaps = gap_words(resume_text, jd_text, background_docs, top_n)
    return {
        "similarity_score": score,
        "match_band": band,
        "gap_words": gaps
    }

# Test all pairs – uncommented
print("\n--- Full analysis results ---")
for key, pair in pairs.items():
    result = analyze_resume_gap(pair["resume"], pair["jd"], background_corpus)
    print(key, result)


--- Full analysis results ---
R1_DataAnalyst {'similarity_score': np.float64(27.83), 'match_band': 'Weak Match', 'gap_words': [('strong', np.float64(0.2262)), ('skills', np.float64(0.2051)), ('aws', np.float64(0.1661)), ('azure', np.float64(0.1661)), ('building', np.float64(0.1661)), ('cloud', np.float64(0.1661)), ('desirable', np.float64(0.1661)), ('etl', np.float64(0.1661))]}
R2_DataScientist {'similarity_score': np.float64(21.06), 'match_band': 'Weak Match', 'gap_words': [('model', np.float64(0.3294)), ('strong', np.float64(0.2243)), ('skills', np.float64(0.2034)), ('aws', np.float64(0.1647)), ('cloud', np.float64(0.1647)), ('deep', np.float64(0.1647)), ('deploying', np.float64(0.1647)), ('docker', np.float64(0.1647))]}
R3_MLEngineer {'similarity_score': np.float64(49.97), 'match_band': 'Moderate Match', 'gap_words': [('build', np.float64(0.1785)), ('cloud', np.float64(0.1785)), ('deploy', np.float64(0.1785)), ('distributed', np.float64(0.1785)), ('frameworks', np.float64(0.1785)),

In [7]:
# TASK 6: Streamlit UI skeleton (minimal – Day 199 will expand)
# Goal: Provide a working stub that imports the engine and displays results.
# Note: Run with `streamlit run this_notebook.py` – but it's only a skeleton.

"""
import streamlit as st
from resumegap_engine import analyze_resume_gap  # will be created Day 199

st.title("ResumeGapAI")
st.write("Paste your resume and job description below to see match score and gap words.")

resume_text = st.text_area("Resume", height=200)
jd_text = st.text_area("Job Description", height=200)

if st.button("Analyze"):
    if resume_text and jd_text:
        result = analyze_resume_gap(resume_text, jd_text, background_corpus)
        st.metric("Similarity Score", f"{result['similarity_score']:.2f}%")
        st.write(f"**Match Band:** {result['match_band']}")
        st.write("**Gap Words (add these to your resume):**")
        for word, weight in result['gap_words']:
            st.write(f"- {word} (weight: {weight})")
    else:
        st.warning("Please fill in both fields.")
"""

'\nimport streamlit as st\nfrom resumegap_engine import analyze_resume_gap  # will be created Day 199\n\nst.title("ResumeGapAI")\nst.write("Paste your resume and job description below to see match score and gap words.")\n\nresume_text = st.text_area("Resume", height=200)\njd_text = st.text_area("Job Description", height=200)\n\nif st.button("Analyze"):\n    if resume_text and jd_text:\n        result = analyze_resume_gap(resume_text, jd_text, background_corpus)\n        st.metric("Similarity Score", f"{result[\'similarity_score\']:.2f}%")\n        st.write(f"**Match Band:** {result[\'match_band\']}")\n        st.write("**Gap Words (add these to your resume):**")\n        for word, weight in result[\'gap_words\']:\n            st.write(f"- {word} (weight: {weight})")\n    else:\n        st.warning("Please fill in both fields.")\n'


<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 4 — Scoring Rubric (100 pts)</h2>
</div>

| Task | Points | Criteria |
|---|---|---|
| T1 clean_text | 15 | Exact match on cleaned R1 resume text |
| T2 naive similarity | 20 | All 3 scores match answer key to 2 decimals |
| T3 fixed similarity + code | 25 | All 3 scores match; correct rows used (-2, -1) |
| T3b interpretation | 10 | States the IDF-spread mechanism, not just "the fix worked" |
| T4 gap_words | 15 | Correct top-8 words + weights for all 3 pairs |
| T5 analyze_resume_gap | 15 | Correct dict structure and values, band boundaries exact (>=70 / >=45) |

**Communication vs technical split applies as always:** a correct score with a hand-wavy T3b explanation
is a communication deduction, not a technical one — flag it as such if it happens.



### Interview framing
*"How would you explain this tool's core scoring logic to a client or interviewer?"*

"I built a resume-to-JD gap analyzer using TF-IDF cosine similarity. The interesting engineering decision was that naive TF-IDF between just two documents underestimates similarity, because IDF has almost no dynamic range with only 2 documents — so I fit the vectorizer against a small background corpus of generic resume/JD language instead, which gave IDF a realistic vocabulary to weight against and produced meaningfully different, more accurate match scores."
